<a href="https://colab.research.google.com/github/MeenakshiRajpurohit/CMPE-297-Natural-Language-Processing/blob/main/HW5_Meenakshi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <span style="color:blue">ChatBot for Registration 2026</span>

In [25]:
import numpy as np
from sklearn.manifold import TSNE
from ast import literal_eval
import matplotlib.pyplot as plt
import json

import os
from dotenv import load_dotenv

import openai
from openai import OpenAI

In [26]:
import os
from dotenv import load_dotenv
os.chdir("..")
print(os.getcwd())
load_dotenv()

/


False

In [27]:
import os
from google.colab import userdata
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()
print("OpenAI client ready")

OpenAI client ready


In [28]:
import os
from dotenv import load_dotenv
import json
from openai import OpenAI

# Load environment variables
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("Warning: OPENAI_API_KEY environment variable not found. Please ensure you have a .env file with OPENAI_API_KEY='your-key'")
    # Optionally fallback or prompt for manual input if needed

# Initialize the client
client = OpenAI(api_key=api_key)

In [29]:
#%pip install resend

In [30]:
MODEL = "gpt-4o-mini"

from openai import OpenAI
client = OpenAI()

def response_request(
    messages,
    model=MODEL,
    tools=None,
    tool_choice="auto",
    conversation=None,
    max_output_tokens=None,
    temperature=None,   # will be ignored for gpt-5 models
):
    if str(model).startswith("gpt-5"):
        # Do NOT send temperature for gpt-5 family
        return client.responses.create(
            model=model,
            input=messages,
            tools=tools,
            tool_choice=tool_choice,
            conversation=conversation,
            max_output_tokens=max_output_tokens,
        )
    else:
        # Non–gpt-5 models can receive temperature
        return client.responses.create(
            model=model,
            input=messages,
            tools=tools,
            tool_choice=tool_choice,
            conversation=conversation,
            max_output_tokens=max_output_tokens,
            temperature=temperature,
        )

## 2. New ChatBot with Function calling
Please note the latest support model, it supports ` gpt-4o-mini, gpt-5-mini`

#### The following is demo code using function calling after the ChatBot decide to send out an email

#### Define a User Defined Function, to send email, if no errors, return `success`, else return `error`

In [34]:
!pip install -q resend

import os, resend
from google.colab import userdata

resend.api_key = userdata.get("resend_api_key")
print("Resend ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.3/126.3 kB 3.0 MB/s eta 0:00:00
Resend ready


In [35]:
from html import escape

def send_confirm_email(to, subject, body):

    # Sender must use a verified domain for real recipients
    SENDER = "SJSU NLP ChatBot <onboarding@resend.dev>"

    # HTML-escape user-provided text
    BODY_TEXT = str(body)
    BODY_HTML_TEXT = escape(BODY_TEXT).replace("\n", "<br>")

    BODY_HTML = f"""
<html>
<head>
    <style>
        body {{
            font-family: Arial, sans-serif;
        }}
        h1 {{
            color: #4CAF50;
        }}
        p {{
            font-size: 14px;
            color: #333333;
        }}
        .registered-courses {{
            font-weight: bold;
            color: #2F4F4F;
        }}
        .homework {{
            margin-top: 20px;
            font-size: 13px;
            color: #555555;
        }}
        a {{
            color: #1E90FF;
            text-decoration: none;
        }}
    </style>
</head>
<body>
    <h1>Congratulations! Registration was successful!</h1>

    <p>This email was sent by
        <a href="https://sjsu.edu">SJSU Fall Class</a>
        using a ChatBot.
    </p>

    <p class="registered-courses">
        Registered Courses:<br>
        {BODY_HTML_TEXT}
    </p>

    <p class="homework">
        You might want to try a similar thing as homework.
    </p>
</body>
</html>
"""

    try:
        response = resend.Emails.send({
            "from": SENDER,
            "to": [to],
            "subject": subject,
            "html": BODY_HTML,
            "text": BODY_TEXT
        })

        print("Email submitted:", response)
        return "success"

    except Exception as e:
        print("Email sending failed:", e)
        return "error"


**User Defined Function**

To refine your approach for developing utilities to interact with a Chat Completions API and manage conversation state, you could declare User Defined Function.

In [36]:
#Step 1, Define UDFs
tools = [
    {
        "type": "function",
        "name": "send_confirm_email",
        "description": "Send an email as confirmation email to student.",
        "parameters": {
            "type": "object",
            "properties": {
                "to": {
                    "type": "string",
                    "description": "the recipient email address",
                    },
                "subject": {
                    "type": "string",
                    "description": "the subject of the email, it should be Registration was successful!",
                    },
                "body": {
                    "type": "string",
                    "description": "the body of the email, contain the content of student course selection",
                    },
                },
            "required": ["to", "subject", "body"],
        },
    }
]

## 2.1 ChatBot Code Snippet Using Loop

In [37]:
chatContext = [
    {'role':'system', 'content':  (
        "Identity\n"
        "You are a smart, friendly virtual assistant that helps students register for Fall 2026 courses. "
        "Be concise, helpful, and polite.\n"
        "Greet student first. \n"
        "- Complete registrations for Fall 2026.\n"
        "- Confirm selections; require email for confirmation.\n"
        "- Detect/resolve time conflicts before finalizing.\n"
        "- If you already greeted, don't greet again; continue the flow.\n\n"
        "Scope: Only Fall 2026 registration. If asked about other terms, apologize and say only Fall 2026 is open.\n"
        "Context Fall 2026 Course List (authoritative source for this assistant)\n"
        "ISE 201, Math Foundations for Decision & Data Sciences — Sat 09:30–11:30 — Instructor: Jacob Sterling \n"
        "CMPE 252, AI and Data Engineering — Mon 19:30–21:00; Sat 15:10–17:10 — Instructor: Emily Clarkson\n"
        "CMPE 180A, Data Structures & Algorithms in Python — Tuesday 19:30–21:00; Sat 13:30–15:00 — Instructor: Nora Kensington\n"
        "CMPE 259, Special Topics: Natural Language Processing — Tuesday 17:30–21:30 — Instructor: Michael Hu\n"
        "CMPE 256, Web and Big Data Mining — Thu 19:30–21:30 — Instructor: Lien Nguyen\n"
        "CMPE 249, Intelligent Autonomous Systems — Fri 19:30–21:30 — Instructor: Yuna Tanaka\n"
    )},
]
chatContext

[{'role': 'system',
  'content': "Identity\nYou are a smart, friendly virtual assistant that helps students register for Fall 2026 courses. Be concise, helpful, and polite.\nGreet student first. \n- Complete registrations for Fall 2026.\n- Confirm selections; require email for confirmation.\n- Detect/resolve time conflicts before finalizing.\n- If you already greeted, don't greet again; continue the flow.\n\nScope: Only Fall 2026 registration. If asked about other terms, apologize and say only Fall 2026 is open.\nContext Fall 2026 Course List (authoritative source for this assistant)\nISE 201, Math Foundations for Decision & Data Sciences — Sat 09:30–11:30 — Instructor: Jacob Sterling \nCMPE 252, AI and Data Engineering — Mon 19:30–21:00; Sat 15:10–17:10 — Instructor: Emily Clarkson\nCMPE 180A, Data Structures & Algorithms in Python — Tuesday 19:30–21:00; Sat 13:30–15:00 — Instructor: Nora Kensington\nCMPE 259, Special Topics: Natural Language Processing — Tuesday 17:30–21:30 — Instruc

In [38]:
#print("SchoolBot: Welcome to the 2026 Courses Registration. What can I do for you?")

In [39]:
# ----- get first user input BEFORE first model call -----
#user_input = input("User: ").strip()
#chatContext.append({"role": "user", "content": user_input})

In [40]:
import json

continue_chat = True

MAX_TOOL_ROUNDS = 5   # cap on tool-calling *rounds* per assistant turn

while continue_chat:
    # 1) Model step
    resp = response_request(chatContext, temperature=0, tools=tools, tool_choice="auto")
    chatContext += resp.output

    # 2) Tool loop: keep resolving tool calls until none remain (but capped)
    for round_idx in range(MAX_TOOL_ROUNDS):
        tool_calls = [it for it in resp.output if it.type == "function_call"]

        if not tool_calls:
            # No tool calls => print assistant text and move on
            print("SchoolBot:", resp.output_text)
            break

        # Execute all tool calls returned in this response
        for item in tool_calls:
            try:
                if item.name == "send_confirm_email":
                    print("System GPT to call! function:", item.name)
                    result = send_confirm_email(**json.loads(item.arguments))
                    print(f"system: Oh, just found the email sending status is {result}")

                    chatContext.append({
                        "type": "function_call_output",
                        "call_id": item.call_id,
                        "output": json.dumps({"send_confirm_email": result})
                    })
                else:
                    # If you have other tools, handle them here
                    chatContext.append({
                        "type": "function_call_output",
                        "call_id": item.call_id,
                        "output": json.dumps({"error": f"Unknown tool: {item.name}"})
                    })

            except Exception as e:
                # Prevent crashes from creating infinite retries
                chatContext.append({
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": json.dumps({"error": f"Tool execution failed: {type(e).__name__}: {e}"})
                })

        # Ask model to continue after tool outputs
        resp = client.responses.create(
            model=MODEL,
            input=chatContext,
            tools=tools,
            tool_choice="auto",
            temperature=0,
        )
        chatContext += resp.output

    else:
        # If we exit the for-loop normally, we hit the cap without finishing
        print(f"system: Tool loop exceeded {MAX_TOOL_ROUNDS} rounds; forcing a final text answer.")
        chatContext.append({
            "role": "user",
            "content": "Stop calling tools now. Give me your best final text answer with what you have."
        })
        resp = client.responses.create(
            model=MODEL,
            input=chatContext,
            tools=tools,
            tool_choice="none",   # force no more tool calls
            temperature=0,
        )
        chatContext += resp.output
        print("SchoolBot:", resp.output_text)

    # 3) Next user input
    user_input = input("User: ").strip()
    chatContext.append({"role": "user", "content": user_input})

    if "bye" in user_input.lower():
        continue_chat = False


SchoolBot: Hello! How can I assist you with your Fall 2026 course registration today?
User: yes
SchoolBot: Great! Please let me know which courses you would like to register for in Fall 2026. Here’s the list:

1. **ISE 201**: Math Foundations for Decision & Data Sciences — Sat 09:30–11:30
2. **CMPE 252**: AI and Data Engineering — Mon 19:30–21:00; Sat 15:10–17:10
3. **CMPE 180A**: Data Structures & Algorithms in Python — Tue 19:30–21:00; Sat 13:30–15:00
4. **CMPE 259**: Special Topics: Natural Language Processing — Tue 17:30–21:30
5. **CMPE 256**: Web and Big Data Mining — Thu 19:30–21:30
6. **CMPE 249**: Intelligent Autonomous Systems — Fri 19:30–21:30

Let me know your selections!
User: **CMPE 259**: Special Topics: Natural Language Processing — Tue 17:30–21:30
SchoolBot: You've selected **CMPE 259: Special Topics: Natural Language Processing** on Tuesday from 17:30 to 21:30. 

Now, let's check for any time conflicts with other courses. Would you like to add more courses, or should I

In [42]:
!pip install -q openai ddgs

# ================= HW5: ChatBot with function calling (my own live web_search tool) =================
import os, json, time
from datetime import datetime, date
import pandas as pd
from openai import OpenAI
from ddgs import DDGS
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")   # Colab secret named OPENAI_API_KEY
client = OpenAI()
MODEL = "gpt-4o-mini"
TODAY = date.today().strftime("%B %d, %Y")
TOOL_LOG = []   # one row per real execution of web_search()


# ---------- 1) MY OWN TOOL: live web search (DuckDuckGo, no extra API key) ----------
def web_search(query, search_type="web", timelimit="none", max_results=5):
    query = (query or "").strip()
    max_results = max(1, min(int(max_results or 5), 8))
    tl = None if timelimit in (None, "", "none") else timelimit
    start, raw, used, error = time.time(), [], search_type, None
    for attempt in range(2):                                   # one retry if rate-limited
        try:
            d = DDGS()
            if search_type == "news":
                raw = d.news(query, max_results=max_results, region="us-en",
                             timelimit=tl if tl in ("d", "w", "m") else None)
                if not raw:                                    # no news -> fall back to web
                    raw, used = d.text(query, max_results=max_results, timelimit=tl, region="us-en"), "web (fallback)"
            else:
                raw = d.text(query, max_results=max_results, timelimit=tl, region="us-en")
            error = None
            break
        except Exception as e:
            error = f"{type(e).__name__}: {e}"
            time.sleep(1.5)
    results = [{"id": i, "title": r.get("title", ""), "url": r.get("href") or r.get("url", ""),
                "snippet": (r.get("body") or "")[:500], "date": r.get("date", "")}
               for i, r in enumerate(raw or [], start=1)]
    TOOL_LOG.append({"time": datetime.now().strftime("%H:%M:%S"), "query": query, "search_type": used,
                     "num_results": len(results), "seconds": round(time.time() - start, 2),
                     "error": None if results else error})
    if not results:
        return {"query": query, "error": error or "No results found", "results": []}
    return {"query": query, "search_type": used, "num_results": len(results), "results": results}


# ---------- 2) Describe the tool to the model ----------
tools = [{
    "type": "function",
    "name": "web_search",
    "description": ("Search the live web. Use for news, current events, recent results, latest versions, "
                    "or facts that may have changed. Do NOT use for greetings, math, or general knowledge."),
    "strict": True,
    "parameters": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Short search query (2-10 words)."},
            "search_type": {"type": "string", "enum": ["web", "news"], "description": "'news' for recent headlines."},
            "timelimit": {"type": "string", "enum": ["d", "w", "m", "y", "none"], "description": "Past day/week/month/year or 'none'."},
            "max_results": {"type": "integer", "description": "1-8 results. Use 5 if unsure."},
        },
        "required": ["query", "search_type", "timelimit", "max_results"],
        "additionalProperties": False,
    },
}]
TOOL_REGISTRY = {"web_search": web_search}

SYSTEM_PROMPT = f"""You are SearchBot, a friendly, concise assistant. Today's date is {TODAY}.
Use the web_search tool for news, recent events, or facts that may have changed. Do not search for greetings,
math, or questions you can answer from the conversation so far. After searching, use ONLY facts from the results,
cite them as [1], [2], and end with a "Sources:" list of URLs. If results don't answer the question, say so.
Keep answers under about 150 words."""


# ---------- 3) Orchestration: model decides -> my code runs the tool -> model answers ----------
def new_chat():
    return [{"role": "developer", "content": SYSTEM_PROMPT}]


def chat_turn(chat, user_input, max_rounds=4):
    chat.append({"role": "user", "content": user_input})
    n_calls = 0
    for _ in range(max_rounds):
        resp = client.responses.create(model=MODEL, input=chat, tools=tools, tool_choice="auto", temperature=0)
        chat += resp.output
        calls = [it for it in resp.output if it.type == "function_call"]
        if not calls:
            return resp.output_text, n_calls                    # final grounded answer
        for item in calls:
            print(f"   [model -> function_call] {item.name}({item.arguments})")
            try:
                result = TOOL_REGISTRY[item.name](**json.loads(item.arguments))
            except Exception as e:
                result = {"error": f"{type(e).__name__}: {e}"}
            n_calls += 1
            print(f"   [my code ran {item.name}] {result.get('num_results', 0)} live results"
                  + (f" | error: {result['error']}" if result.get("error") else ""))
            chat.append({"type": "function_call_output", "call_id": item.call_id, "output": json.dumps(result)})
    resp = client.responses.create(model=MODEL, input=chat, tools=tools, tool_choice="none", temperature=0)
    chat += resp.output
    return resp.output_text, n_calls


# ---------- 4) Demo with new examples ----------
chat2 = new_chat()
new_examples = [
    "What is 15% of 240?",                                                         # no search expected
    "What are the biggest AI news stories this week?",                             # news search
    "Who won the most recent NBA championship, and who was the Finals MVP?",       # web search
    "Of the AI stories you found, which one matters most for students? Answer in 2 sentences.",  # from context
]

for user_msg in new_examples:
    print("=" * 100)
    print("User:", user_msg)
    answer, n_calls = chat_turn(chat2, user_msg)
    tag = f"used web_search x{n_calls}" if n_calls else "no tool call needed"
    print(f"\nSearchBot ({tag}):\n{answer}\n")

print("=" * 100)
print(f"Proof: web_search() was executed {len(TOOL_LOG)} time(s) by my code:")
display(pd.DataFrame(TOOL_LOG))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 37.8 MB/s eta 0:00:00
User: What is 15% of 240?

SearchBot (no tool call needed):
15% of 240 is 36.

User: What are the biggest AI news stories this week?
   [model -> function_call] web_search({"query":"AI news","search_type":"news","timelimit":"w","max_results":5})
   [my code ran web_search] 5 live results

SearchBot (used web_search x1):
Here are some of the biggest AI news stories from this week:

1. **Investors Concerned Over AI Debt**: A Financial Times podcast discusses investor worries regarding the financial implications of AI investments. [Read more](https://www.ft.com/content/332bc215-532e-4c53-bdbd-6e3d5311bd5e?syn-25a6b1a6=1) (Oct 16).

2. **Databricks Expands in Manhattan**: The AI company Databricks is doubling its office space at 5 Bryant Park, reflecting confidence in AI development in New York City. [Read more](https://nypost.com/2026/1

,time,query,search_type,num_results,seconds,error
0,02:42:08,AI news,news,5,0.60,None
1,02:42:20,NBA championship winner 2026,news,5,0.25,None
2,02:42:21,NBA Finals 2026 winner MVP,news,5,0.24,None
